# Part A: Airbnb Price Prediction
### Machine Learning Project – Regression
---

## 1. Understanding the Problem

**What are we trying to solve?**

Airbnb hosts often struggle to price their listings correctly. If the price is too high, guests may not book. If it's too low, the host loses money.

In this project, we will build a **regression model** that predicts the **price of an Airbnb listing** based on features like:
- Room type and property type
- Location (city)
- Number of reviews and ratings
- Number of bedrooms, bathrooms, beds
- Host characteristics

**Target Variable:** `log_price` (the log of the listing price – already transformed in the dataset)

**Type of Problem:** Regression (predicting a continuous value)

## 2. Import Libraries

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print("All libraries imported successfully!")

## 3. Load and Explore the Dataset

In [ ]:
# Load the dataset
df = pd.read_excel('Airbnb_data.xlsx')

print("Dataset loaded successfully!")
print(f"Shape: {df.shape[0]} rows and {df.shape[1]} columns")

In [ ]:
# View the first few rows
df.head()

In [ ]:
# Check column names and data types
df.info()

In [ ]:
# Basic statistics of numerical columns
df.describe()

In [ ]:
# Check for missing values in each column
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing Percentage (%)': missing_pct
})

# Show only columns with missing values
print("Columns with missing values:")
missing_df[missing_df['Missing Count'] > 0]

### 3.1 Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of the target variable (log_price)
plt.figure(figsize=(10, 5))
sns.histplot(df['log_price'], bins=50, kde=True, color='steelblue')
plt.title('Distribution of Log Price (Target Variable)', fontsize=14)
plt.xlabel('Log Price')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print(f"Average log_price: {df['log_price'].mean():.2f}")
print(f"This roughly corresponds to price: ${np.exp(df['log_price'].mean()):.2f}")

In [ ]:
# Average price by Room Type
plt.figure(figsize=(9, 5))
room_avg = df.groupby('room_type')['log_price'].mean().sort_values(ascending=False)
sns.barplot(x=room_avg.index, y=room_avg.values, palette='Blues_d')
plt.title('Average Log Price by Room Type', fontsize=14)
plt.xlabel('Room Type')
plt.ylabel('Average Log Price')
plt.tight_layout()
plt.show()

In [ ]:
# Average price by City
plt.figure(figsize=(10, 5))
city_avg = df.groupby('city')['log_price'].mean().sort_values(ascending=False)
sns.barplot(x=city_avg.index, y=city_avg.values, palette='Greens_d')
plt.title('Average Log Price by City', fontsize=14)
plt.xlabel('City')
plt.ylabel('Average Log Price')
plt.tight_layout()
plt.show()

In [ ]:
# Relationship between number of bedrooms and price
plt.figure(figsize=(10, 5))
bedroom_avg = df.groupby('bedrooms')['log_price'].mean()
sns.lineplot(x=bedroom_avg.index, y=bedroom_avg.values, marker='o', color='coral')
plt.title('Average Log Price vs Number of Bedrooms', fontsize=14)
plt.xlabel('Bedrooms')
plt.ylabel('Average Log Price')
plt.xlim(0, 10)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numerical columns
num_cols = ['log_price', 'accommodates', 'bathrooms', 'number_of_reviews',
            'review_scores_rating', 'bedrooms', 'beds']

plt.figure(figsize=(9, 7))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of Numerical Features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Count of listings by property type (Top 10)
plt.figure(figsize=(12, 5))
top_props = df['property_type'].value_counts().head(10)
sns.barplot(x=top_props.values, y=top_props.index, palette='Purples_r')
plt.title('Top 10 Property Types by Count', fontsize=14)
plt.xlabel('Number of Listings')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# Select useful features for the model
# We drop columns that are IDs, free text, dates, or have too many missing values
selected_features = [
    'log_price',          # Target variable
    'property_type',      # Type of property
    'room_type',          # Entire place / Private room / Shared
    'accommodates',       # Number of guests it accommodates
    'bathrooms',          # Number of bathrooms
    'bedrooms',           # Number of bedrooms
    'beds',               # Number of beds
    'bed_type',           # Type of bed
    'cancellation_policy',# Cancellation policy type
    'city',               # City of the listing
    'number_of_reviews',  # Total reviews received
    'review_scores_rating',# Average review score
    'host_has_profile_pic',# Whether host has profile picture
    'host_identity_verified', # Whether host is verified
    'instant_bookable',   # Whether it can be booked instantly
    'cleaning_fee',       # Whether cleaning fee is charged
]

df_model = df[selected_features].copy()
print(f"Working dataset shape: {df_model.shape}")
df_model.head()

In [ ]:
# Handle missing values

# For numerical columns: fill with the median value
df_model['bathrooms'].fillna(df_model['bathrooms'].median(), inplace=True)
df_model['bedrooms'].fillna(df_model['bedrooms'].median(), inplace=True)
df_model['beds'].fillna(df_model['beds'].median(), inplace=True)
df_model['review_scores_rating'].fillna(df_model['review_scores_rating'].median(), inplace=True)

# For categorical columns: fill with the most common value (mode)
df_model['host_has_profile_pic'].fillna(df_model['host_has_profile_pic'].mode()[0], inplace=True)
df_model['host_identity_verified'].fillna(df_model['host_identity_verified'].mode()[0], inplace=True)

# Check if any missing values remain
print("Missing values after cleaning:")
print(df_model.isnull().sum())

In [ ]:
# Encode categorical columns using Label Encoding
# This converts text categories to numbers (e.g., 'Apartment' → 0, 'House' → 1)

categorical_cols = [
    'property_type', 'room_type', 'bed_type',
    'cancellation_policy', 'city',
    'host_has_profile_pic', 'host_identity_verified', 'instant_bookable'
]

le = LabelEncoder()
for col in categorical_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

# Convert boolean column to integer
df_model['cleaning_fee'] = df_model['cleaning_fee'].astype(int)

print("Categorical encoding done!")
df_model.head()

In [ ]:
# Separate features (X) and target (y)
X = df_model.drop('log_price', axis=1)  # Features
y = df_model['log_price']               # Target variable

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Train-Test Split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]} rows")
print(f"Testing set size : {X_test.shape[0]} rows")

## 5. Model Building

In [ ]:
# --- Model 1: Linear Regression ---
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

print("Linear Regression model trained!")

In [ ]:
# --- Model 2: Decision Tree Regressor ---
dt_model = DecisionTreeRegressor(max_depth=6, random_state=42)
dt_model.fit(X_train, y_train)
dt_preds = dt_model.predict(X_test)

print("Decision Tree model trained!")

In [ ]:
# --- Model 3: Random Forest Regressor (Best model) ---
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

print("Random Forest model trained!")

## 6. Model Evaluation

In [ ]:
# Function to compute and display evaluation metrics
def evaluate_model(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"--- {name} ---")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  MAE  : {mae:.4f}")
    print(f"  R²   : {r2:.4f}")
    print()
    return {'Model': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

results = []
results.append(evaluate_model('Linear Regression', y_test, lr_preds))
results.append(evaluate_model('Decision Tree',     y_test, dt_preds))
results.append(evaluate_model('Random Forest',     y_test, rf_preds))

In [ ]:
# Compare models in a table
results_df = pd.DataFrame(results)
results_df = results_df.set_index('Model')
print("Model Comparison:")
results_df

In [ ]:
# Visualize R² scores for all models
plt.figure(figsize=(8, 5))
sns.barplot(x=results_df.index, y=results_df['R2'], palette='Set2')
plt.title('R² Score Comparison Across Models', fontsize=14)
plt.ylabel('R² Score')
plt.xlabel('Model')
plt.ylim(0, 1)
for i, v in enumerate(results_df['R2']):
    plt.text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted plot for the best model (Random Forest)
plt.figure(figsize=(8, 6))
plt.scatter(y_test, rf_preds, alpha=0.3, color='steelblue', s=10)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Log Price')
plt.ylabel('Predicted Log Price')
plt.title('Random Forest: Actual vs Predicted Price', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance from Random Forest
importances = rf_model.feature_importances_
feat_importance = pd.Series(importances, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=feat_importance.values, y=feat_importance.index, palette='viridis')
plt.title('Feature Importance (Random Forest)', fontsize=14)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print("Top 5 important features:")
print(feat_importance.head())

In [ ]:
# Sample predictions vs actual values
comparison = pd.DataFrame({
    'Actual Log Price': y_test.values[:10],
    'Predicted Log Price': rf_preds[:10],
    'Actual Price ($)': np.exp(y_test.values[:10]).round(2),
    'Predicted Price ($)': np.exp(rf_preds[:10]).round(2)
})
print("Sample Predictions (first 10):")
comparison

## 7. Conclusion

Here is a summary of what we found in this project:

1. **Room type and city matter most** – Entire homes/apartments in high-demand cities like NYC and SF command significantly higher prices than shared or private rooms.

2. **More bedrooms = higher price** – The number of bedrooms, beds, and how many guests a listing can accommodate are strong predictors of price.

3. **Random Forest performed best** – With an R² score significantly higher than Linear Regression or Decision Tree, it captured non-linear relationships in the data effectively.

4. **Review scores have moderate impact** – Listings with better ratings tend to charge slightly more, but it's not the dominant factor compared to size and location.

5. **The model can help hosts price better** – By understanding which features drive price, hosts can make informed decisions to optimize their listing prices and maximize bookings.